# Silver Layer - Ride Events

## Objective

Transform raw ride streaming events into a business-ready dataset for KPI reporting.

### Input

- workspace.default.bronze_ride_events

### 

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    to_date,
    hour,
    current_date
)

bronze_events_df = spark.table(
    "workspace.default.bronze_ride_events"
)

In [0]:
display(
    bronze_events_df.limit(20)
)

In [0]:
bronze_events_df.printSchema()

## Apply Data Quality Rules

Keep only valid ride events.

In [0]:
silver_events_df = (
    bronze_events_df
    .filter(
        col("ride_id").isNotNull()
    )
    .filter(
        col("event_type").isin(
            "REQUESTED",
            "STARTED",
            "COMPLETED"
        )
    )
)

## Derive Ride Status

Convert technical event types into business-friendly ride statuses.

In [0]:
silver_events_df = (
    silver_events_df
    .withColumn(
        "ride_status",
        when(
            col("event_type") == "REQUESTED",
            "Pending"
        )
        .when(
            col("event_type") == "STARTED",
            "In Progress"
        )
        .otherwise(
            "Completed"
        )
    )
)

## Create Completion Indicator

Derive a reusable flag for completion-related KPIs.

In [0]:
silver_events_df = (
    silver_events_df
    .withColumn(
        "is_completed",
        when(
            col("event_type") == "COMPLETED",
            1
        ).otherwise(0)
    )
)

## Create Analytical Date Dimensions

Add reporting-friendly date attributes.

In [0]:
silver_events_df = (
    silver_events_df
    .withColumn(
        "event_date",
        to_date(col("timestamp"))
    )
    .withColumn(
        "event_hour",
        hour(col("timestamp"))
    )
    .withColumn(
        "processing_date",
        current_date()
    )
)

## Review Enriched Dataset

In [0]:
display(
    silver_events_df.limit(50)
)

## Create Silver Table

In [0]:
silver_events_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .saveAsTable(
        "workspace.default.silver_ride_events"
    )

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.silver_ride_events

In [0]:
%sql
SELECT
    ride_status,
    COUNT(*) AS total_events
FROM workspace.default.silver_ride_events
GROUP BY ride_status
ORDER BY total_events DESC

In [0]:
%sql
SELECT
    SUM(is_completed) AS completed_events
FROM workspace.default.silver_ride_events

In [0]:
%sql
SELECT
    event_hour,
    COUNT(*) AS total_events
FROM workspace.default.silver_ride_events
GROUP BY event_hour
ORDER BY event_hour

## Business Value Added

### Bronze Layer

- Raw streaming ride events
- Technical event types

### Silver Layer

- Validated ride events
- Business-friendly ride statuses
- Completion indicators
- Reporting date dimensions
- Processing metadata

This layer prepares ride events for Gold KPI generation.